In [1]:
import boto3
import torch

from PIL import Image
from io import BytesIO
from torchvision import transforms

session = boto3.Session(profile_name="ATLAS")
s3 = session.client("s3", region_name="us-east-2")

SOURCE_BUCKET = "atlas-acm-research-f26"
SOURCE_PREFIX = "satellite/raw/"

print("AWS connected successfully")

def load_rgb_image(key):
    response = s3.get_object(Bucket=SOURCE_BUCKET, Key=key)
    try:
        with Image.open(BytesIO(response["Body"].read())) as image:
            return image.convert("RGB")
    finally:
        response["Body"].close()


AWS connected successfully


In [2]:
#retrieve image from S3
image_keys = []

paginator = s3.get_paginator("list_objects_v2")

for page in paginator.paginate(
    Bucket=SOURCE_BUCKET,
    Prefix=SOURCE_PREFIX
):
    for obj in page.get("Contents", []):
        key = obj["Key"]

        if key.lower().endswith(".png"):
            image_keys.append(key)

image_keys.sort()

print("Total images found:", len(image_keys))
print("First 5 images:", image_keys[:5])

Total images found: 180
First 5 images: ['satellite/raw/0007805491.png', 'satellite/raw/0014593270.png', 'satellite/raw/0014930068.png', 'satellite/raw/0014997588.png', 'satellite/raw/0015884451.png']


In [3]:
#image to tensor preprocessing
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

In [4]:
#download and convert images 
key = image_keys[0]

image = load_rgb_image(key)

tensor = transform(image)

image_id = key.rsplit("/", 1)[-1].removesuffix(".png")

print("Dataset ID:", image_id)
print("Original image size:", image.size)
print("Tensor shape:", tensor.shape)
print("Tensor dtype:", tensor.dtype)
print("Tensor range:", tensor.min().item(), tensor.max().item())

Dataset ID: 0007805491
Original image size: (1000, 1000)
Tensor shape: torch.Size([3, 224, 224])
Tensor dtype: torch.float32
Tensor range: 0.20392157137393951 0.9921568632125854


In [5]:
DEST_BUCKET = "atlas-acm-research-f26"
DEST_PREFIX = "satellite/processed/"

In [6]:

uploaded = 0
failed = []

for i, key in enumerate(image_keys, start=1):
    image_id = key.rsplit("/", 1)[-1].removesuffix(".png")
    tensor_key = f"{DEST_PREFIX}{image_id}.pt"

    try:
        # download PNG from source S3 bucket
        image = load_rgb_image(key)

        # convert image into PyTorch tensor
        tensor = transform(image)

        # serialize tensor in memory
        buffer = BytesIO()
        torch.save({
            "id": image_id,
            "image": tensor,
            "source_key": key
        }, buffer)

        buffer.seek(0)

        # upload .pt file to destination S3 bucket
        s3.upload_fileobj(
            buffer,
            DEST_BUCKET,
            tensor_key
        )

        uploaded += 1
        print(f"[{i}/{len(image_keys)}] Uploaded: {tensor_key}")

    except Exception as e:
        failed.append((image_id, str(e)))
        print(f"[{i}/{len(image_keys)}] ERROR {image_id}: {e}")

print(f"\nSuccessfully uploaded: {uploaded}")
print(f"Failed: {len(failed)}")

[1/180] Uploaded: satellite/processed/0007805491.pt
[2/180] Uploaded: satellite/processed/0014593270.pt
[3/180] Uploaded: satellite/processed/0014930068.pt
[4/180] Uploaded: satellite/processed/0014997588.pt
[5/180] Uploaded: satellite/processed/0015884451.pt
[6/180] Uploaded: satellite/processed/0016282459.pt
[7/180] Uploaded: satellite/processed/0018518989.pt
[8/180] Uploaded: satellite/processed/0025706343.pt
[9/180] Uploaded: satellite/processed/0028544618.pt
[10/180] Uploaded: satellite/processed/0029827954.pt
[11/180] Uploaded: satellite/processed/0031800424.pt
[12/180] Uploaded: satellite/processed/0032560290.pt
[13/180] Uploaded: satellite/processed/0034770750.pt
[14/180] Uploaded: satellite/processed/0052227492.pt
[15/180] Uploaded: satellite/processed/0079818498.pt
[16/180] Uploaded: satellite/processed/0095349317.pt
[17/180] Uploaded: satellite/processed/0122665255.pt
[18/180] Uploaded: satellite/processed/0128499684.pt
[19/180] Uploaded: satellite/processed/0131891022.pt
[2